In [ ]:
import pandas as pd
import numpy as np
from scipy.cluster import hierarchy
from scipy.spatial.distance import squareform
from collections import defaultdict

def get_correlated_clusters(path, corr_threshold=0.85, target_col='TARGET', base=None):
    """
    Finds clusters of highly correlated features in a dataframe.
    """
    df = pd.read_parquet(path)

    if base is not None and target_col not in df.columns:
        df = df.merge(base, on='SK_ID_CURR', how='inner')

    num_cols = df.select_dtypes(include=np.number).columns.tolist()
    if target_col in num_cols:
        num_cols.remove(target_col)

    df = df[num_cols]  # Only keep numeric columns for correlation analysis

    print("1. Calculating Spearman correlation matrix...")
    # .abs() ensures we group strong negative correlations too (e.g., -0.90)
    corr_matrix = df.corr(method='spearman').abs()
    
    print("2. Converting correlation to distance...")
    # Distance = 1 - correlation
    distance_matrix = 1 - corr_matrix

    # 1. Force perfect symmetry by averaging the matrix with its transpose
    distance_matrix = (distance_matrix + distance_matrix.T) / 2

    # 2. Ensure the diagonal is exactly 0
    np.fill_diagonal(distance_matrix.values, 0)
    
    # Convert to condensed 1D array for the clustering algorithm
    condensed_dist = squareform(distance_matrix)
    
    print("3. Performing hierarchical clustering...")
    # 'ward' linkage minimizes the variance within each cluster
    linkage_matrix = hierarchy.linkage(condensed_dist, method='ward')
    
    print(f"4. Isolating clusters with > {corr_threshold} correlation...")
    # Distance threshold is the inverse of the correlation threshold
    distance_threshold = 1 - corr_threshold
    cluster_labels = hierarchy.fcluster(linkage_matrix, distance_threshold, criterion='distance')
    
    # Map features to their assigned cluster IDs
    cluster_dict = defaultdict(list)
    for feature, cluster_id in zip(corr_matrix.columns, cluster_labels):
        cluster_dict[cluster_id].append(feature)
        
    # Filter out "clusters" that only contain 1 feature (no high correlation)
    correlated_clusters = {k: v for k, v in cluster_dict.items() if len(v) > 1}
    
    print(f"Found {len(correlated_clusters)} clusters of highly correlated features.")
    return correlated_clusters

# --- Example Usage ---
# df_features = X_train.sample(10000) # Tip: Sample your data to make the matrix calculation faster!
# clusters = get_correlated_clusters(df_features, corr_threshold=0.85)

In [ ]:
correlated_metrics_list = []

for label, filename in list(datasets.items())[:2]:
    path = os.path.join(INTERMEDIATE, filename)
    metrics = get_correlated_clusters(path=path, base=base)
    correlated_metrics_list.append(metrics)

print("\nAll datasets processed.")


All datasets processed.
